# O+H2 reruns (oh2_escape_rerun): convergence, profiles, escape, wave response

Reads the analysis files (`outputs/oh2_escape_rerun/analysis/*_analysis.nc`) written by `compute_analysis.py`,
which reads the run files (`<Results>/<alias>_run.nc`) built by `compile_netcdf.py`. Data tiers and naming: `docs/CONVENTIONS.md`. Figures go to `outputs/oh2_escape_rerun/figures/`.

Runs: **OH2_A** (128 cells, flux wave from t = 9,849 s, ~4.2 periods) and **OH2_B_r1-r3**
(35 cells, no wave, three independent replicates). Colours: O blue, H2 orange (validated
categorical pair); references grey.

In [1]:
import warnings; warnings.simplefilter("ignore", RuntimeWarning)
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt
from dsmclib.paths import output_dir
from dsmclib.ncrun import open_run_nc
from dsmclib import wave

CAMPAIGN = "oh2_escape_rerun"
ANALYSIS, FIG, TAB = (output_dir(CAMPAIGN, k) for k in ("analysis", "figures", "tables"))
RUNS = ["OH2_A", "OH2_B_r1", "OH2_B_r2", "OH2_B_r3"]
B = RUNS[1:]
AN = {r: xr.open_dataset(ANALYSIS / f"{r}_analysis.nc").load() for r in RUNS}
SUM = xr.open_dataset(ANALYSIS / "oh2_escape_rerun_analysis.nc").load()
COL = {"O": "#2a78d6", "H2": "#eb6834"}
REF = "#6b6b68"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.grid": True, "grid.color": "#e4e3df",
                     "grid.linewidth": 0.6, "axes.edgecolor": "#8a8984", "axes.spines.top": False,
                     "axes.spines.right": False, "lines.linewidth": 2, "font.size": 10,
                     "legend.frameon": False})
win = (AN["OH2_A"].attrs["profile_window_t0"], AN["OH2_A"].attrs["profile_window_t1"])
print("profile window", win)

profile window (np.float64(6849.395233571811), np.float64(9849.395233571811))


## 1. Approach to steady state
Domain inventory per species. O levels off within a few hundred seconds; H2 keeps rising. Shaded: profile window; dashed: OH2_A wave start.

In [2]:
conv = pd.read_csv(TAB / "convergence_summary.csv"); display(conv.round(4))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for ax, sp in zip(axes, ["O", "H2"]):
    for r in RUNS:
        n = AN[r].N_phys.sel(species=sp)
        ax.plot(n.time / 1e3, n, color=COL[sp], lw=2 if r == "OH2_A" else 1, alpha=1 if r == "OH2_A" else 0.55,
                ls="-" if r == "OH2_A" else "--", label="OH2_A (128 cells)" if r == "OH2_A" else ("OH2_B r1-r3 (35 cells)" if r == "OH2_B_r1" else None))
    ax.axvspan(win[0] / 1e3, win[1] / 1e3, color=REF, alpha=0.12, lw=0)
    ax.axvline(AN["OH2_A"].attrs["pre_forcing_end"] / 1e3, color=REF, ls=":", lw=1.2)
    row = conv[(conv.run == "OH2_A") & (conv.species == sp)].iloc[0]
    status = f"steady after {row.t_steady_start_s:.0f} s" if row.steady else f"not steady: +{row.drift_pct_per_ks:.1f}% per ks"
    ax.set_title(f"{sp}: molecules in domain ({status}, OH2_A)", fontsize=10, loc="left")
    ax.set_xlabel("time [ks]"); ax.set_ylabel("N [molecules]")
    ax.legend(loc="lower right")
fig.tight_layout(); fig.savefig(FIG / "fig1_inventory.png"); plt.show()

,run,species,steady,t_steady_start_s,drift_pct_per_ks,rel_drift_ub,N_phys_end,fill_tau_s,fill_tau_err_s,fill_tau_over_span,t_pre_end_s
0,OH2_A,O,True,206.7657,0.1685,0.0408,6.255671e+34,8.770000e+01,7.533080e+01,0.0099,9849.3952
1,OH2_A,H2,False,NaN,6.0872,0.1427,5.274992e+32,7.798798e+07,3.950516e+09,8812.8028,9849.3952
2,OH2_B_r1,O,True,281.9982,0.0934,0.0236,6.438993e+34,9.823150e+01,7.438890e+01,0.0085,12595.2580
3,OH2_B_r1,H2,False,NaN,4.7020,0.1293,5.789034e+32,3.998741e+04,9.482755e+03,3.4486,12595.2580
4,OH2_B_r2,O,True,281.9982,0.0456,0.0131,6.418319e+34,1.383768e+02,9.096510e+01,0.0119,12595.2580
5,OH2_B_r2,H2,False,NaN,4.7590,0.1297,6.020196e+32,4.459501e+05,1.029354e+06,38.4597,12595.2580
6,OH2_B_r3,O,True,281.9982,0.0815,0.0197,6.448898e+34,1.508966e+02,1.445459e+02,0.0130,12595.2580
7,OH2_B_r3,H2,False,NaN,5.2704,0.1450,6.157216e+32,8.027749e+04,1.806911e+04,6.9233,12595.2580


/var/folders/n6/0qs5g9yx36v79k6jdbtt7b340000gq/T/ipykernel_48856/2606185102.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "fig1_inventory.png"); plt.show()


## 2. Time-mean profiles vs the 270 K isothermal (barometric, 1/r^2 gravity) reference
OH2_A line; B replicate mean with ±1 std band; dashed: isothermal reference anchored at the lowest cell.

In [3]:
a = AN["OH2_A"]; b0 = AN["OH2_B_r1"]
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for sp in ["O", "H2"]:
    alt_a = a.altitude / 1e3; alt_b = b0.altitude / 1e3
    n_a = a.profile_num_dens.sel(species=sp)
    ok = a.profile_num_part.sel(species=sp) > 1
    axes[0].plot(n_a.where(ok), alt_a, color=COL[sp], label=f"{sp} OH2_A")
    m, s = SUM.B_num_dens_mean.sel(species=sp), SUM.B_num_dens_std.sel(species=sp)
    okb = b0.profile_num_part.sel(species=sp) > 1
    axes[0].errorbar(m.where(okb), alt_b.where(okb), xerr=s.where(okb), fmt="s", ms=5, mfc="white", mec=COL[sp],
                     ecolor=COL[sp], lw=1, label=f"{sp} OH2_B mean ±1σ (35-cell centres)")
    axes[0].plot(a.profile_num_dens_isothermal.sel(species=sp), alt_a, color=REF, ls="--", lw=1.2,
                 label="isothermal 270 K (each species)" if sp == "O" else None)
    axes[1].plot(a.profile_num_dens_ratio_isothermal.sel(species=sp).where(ok), alt_a, color=COL[sp], label=sp)
    for r in B:
        okr = AN[r].profile_num_part.sel(species=sp) > 1
        axes[1].plot(AN[r].profile_num_dens_ratio_isothermal.sel(species=sp).where(okr), alt_b, "s", ms=4,
                     mfc="white", mec=COL[sp], alpha=0.8)
    axes[2].plot(a.profile_temp_trans_corr.sel(species=sp).where(ok), alt_a, color=COL[sp], label=sp)
    z = float(a.exobase_altitude.sel(species=sp))
    for ax in axes:
        ax.axhline(z, color=COL[sp], lw=0.8, ls=":")
axes[0].set_xscale("log"); axes[0].set_xlim(1e4, 3e16); axes[0].set_xlabel("n [m$^{-3}$]")
axes[1].set_xscale("log"); axes[1].set_xlim(1e-3, 3); axes[1].axvline(1, color=REF, lw=1); axes[1].set_xlabel("n / n$_{iso}$")
axes[2].axvline(270, color=REF, lw=1, ls="--"); axes[2].set_xlabel("T$_{trans}$ [K]"); axes[2].set_xlim(0, 400)
axes[0].set_yscale("log"); axes[0].set_ylabel("altitude [km]"); axes[0].set_ylim(100, 45000)
axes[0].legend(fontsize=8, loc="upper right"); axes[1].legend(fontsize=8); axes[2].legend(fontsize=8)
axes[0].set_title("number density", loc="left", fontsize=10); axes[1].set_title("ratio to isothermal (A line, B squares)", loc="left", fontsize=10)
axes[2].set_title("translational temperature (OH2_A)", loc="left", fontsize=10)
fig.text(0.01, 0.005, f"Time mean over {win[0]:.0f}-{win[1]:.0f} s; dotted lines: exobase (Kn = 1) per species; cells with < 1 computational particle omitted.", fontsize=8, color=REF)
fig.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(FIG / "fig2_profiles.png"); plt.show()

/var/folders/n6/0qs5g9yx36v79k6jdbtt7b340000gq/T/ipykernel_48856/3108190312.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(FIG / "fig2_profiles.png"); plt.show()


In [4]:
# A vs B (grid resolution) on B's cell centres, collisional region
cmp = pd.DataFrame({"alt_km": SUM.altitude_km.values,
                    "O A/B": SUM.A_over_B_num_dens.sel(species="O").values,
                    "O nsigma": SUM.A_minus_B_num_dens_nsigma.sel(species="O").values,
                    "H2 A/B": SUM.A_over_B_num_dens.sel(species="H2").values,
                    "H2 nsigma": SUM.A_minus_B_num_dens_nsigma.sel(species="H2").values})
display(cmp.head(14).round(3))

,alt_km,O A/B,O nsigma,H2 A/B,H2 nsigma
0,100.772,1.009,1.652,0.999,-0.205
1,102.346,1.001,0.065,0.998,-0.302
2,103.983,1.005,0.632,1.000,0.004
3,105.687,0.997,-0.386,0.997,-0.394
4,107.465,1.003,0.619,1.001,0.087
5,109.321,0.998,-0.283,1.010,0.765
6,111.264,1.001,0.126,1.021,1.022
7,113.302,1.012,1.045,1.047,2.165
8,115.445,1.017,1.161,1.055,1.948
9,117.702,1.008,0.465,1.052,3.243


## 3. Escape
H2 escape-rate estimates (O escape is negligible: λ ≈ 80 at its exobase). The direct DSMC count at the 40,000 km top is zero in every run; the transit-time model shows why (median exobase-to-top travel time > run length).

In [5]:
esc = pd.read_csv(TAB / "escape_summary.csv"); display(esc.round(4).T)
h = esc[esc.species == "H2"].set_index("run")
fig, ax = plt.subplots(figsize=(9, 3.8))
rows = [("Jeans at lower boundary (100 km, setup)", h.jeans_rate_lower_boundary_s, None),
        ("Jeans at DSMC exobase", h.jeans_rate_exobase_s, None),
        ("DSMC top count: 95% upper limit", h.dsmc_escape_ul95_s, None),
        ("mass balance: outflow + 2σ", (h.balance_outflow_s + 2 * h.balance_outflow_se_s).clip(lower=0), None)]
for i, (label, vals, _) in enumerate(rows):
    for j, r in enumerate(RUNS):
        ax.plot(vals[r], i + (j - 1.5) * 0.12, "o", ms=8, color=COL["H2"] if r == "OH2_A" else "#f3a07c",
                mec="white", mew=1.5, label=("OH2_A" if r == "OH2_A" else "OH2_B r1-r3") if i == 0 and j < 2 else None)
ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows]); ax.invert_yaxis()
ax.set_xscale("log"); ax.set_xlabel("H$_2$ escape rate [s$^{-1}$]"); ax.legend(loc="upper right", fontsize=8)
ax.set_title("H2 escape: estimates and bounds (upper limits are bounds, not measurements)", loc="left", fontsize=10)
fig.tight_layout(); fig.savefig(FIG / "fig3_escape.png"); plt.show()
print(h[["expected_npart_at_jeans_exobase", "transit_expected_npart", "transit_median_travel_time_s", "transit_p_zero"]])

,0,1,2,3,4,5,6,7
run,OH2_A,OH2_A,OH2_B_r1,OH2_B_r1,OH2_B_r2,OH2_B_r2,OH2_B_r3,OH2_B_r3
species,O,H2,O,H2,O,H2,O,H2
exobase_km,244.4979,412.3103,241.3072,408.0837,235.7902,403.1033,237.3705,398.0939
n_exobase_m3,327096580936306.4375,2013847954936.793213,342765575949496.0625,1945481714628.593506,357154080120491.625,2052815814924.13208,358476421589998.125,2200666438131.035156
T_exobase_K,284.7568,253.2817,269.6191,251.1331,258.0796,244.6905,255.9328,240.785
jeans_lambda_exobase,79.6437,10.8056,84.1892,10.9101,88.0874,11.2121,88.7876,11.409
jeans_rate_exobase_s,0.0173,35682857513752673006911488.0,0.0002,31121350492576168774991872.0,0.0,24509205121558239833489408.0,0.0,21692745544613990181634048.0
jeans_rate_lower_boundary_s,0.0002,284995387707765990233210880.0,0.0002,284995387707765990233210880.0,0.0002,284995387707765990233210880.0,0.0002,284995387707765990233210880.0
dsmc_escape_npart,0,0,0,0,0,0,0,0
dsmc_escape_rate_s,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


          expected_npart_at_jeans_exobase  transit_expected_npart  \
run                                                                 
OH2_A                            2.210707                0.000856   
OH2_B_r1                         2.528400                0.008586   
OH2_B_r2                         1.991208                0.006047   
OH2_B_r3                         1.762390                0.004977   

          transit_median_travel_time_s  transit_p_zero  
run                                                     
OH2_A                     16915.066156        0.999145  
OH2_B_r1                  18664.470887        0.991451  
OH2_B_r2                  18742.299663        0.993972  
OH2_B_r3                  18790.352234        0.995035  


/var/folders/n6/0qs5g9yx36v79k6jdbtt7b340000gq/T/ipykernel_48856/3744876572.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "fig3_escape.png"); plt.show()


## 4. Wave response (OH2_A)
Flux wave at the lower boundary: the injected flux is modulated as F0 (1 + 0.25 sin ωt) (`MOD_Particle.f90`, `Compute_Flux_Wave_Perturbation`, `AMP_inject` = 0.25), with ω = 9.5e-3 rad/s and P = 661 s. Four whole periods are fitted with a + b·t + A cos ωt + B sin ωt. The background is the pre-forcing profile window, and cells with < 10 background computational particles are masked. δn and δT are relative to the background. δv_r is absolute (m/s), because the background radial velocity is about 0.

**Noise.** The high-frequency scatter is DSMC shot noise, not a parsing artefact; `checks/regress_oh2_wave_series.py` re-parses the raw ASCII and matches it bit for bit. At 100–150 km a cell holds only about 105 O and 40 H2 computational particles, so δn/n̄ scatters by about 1/√N ≈ 0.1 per 4.8 s window. The **noise floor** (shaded in fig4b) is the same fit at ω over the 4 periods *before* the forcing starts, which shows what noise alone produces. The ±1σ errors use AR(1)-corrected OLS. The null test shows they are calibrated for O and for H2 δv_r. For H2 δn and δT they are about 25–30% too small: the median null amp/σ is 1.46–1.52, against 1.18 for Rayleigh noise. So treat H2 2σ cuts as roughly 1.5σ. Phases are shown only where amp > 2σ.

In [6]:
run = open_run_nc("OH2_A")
fw = wave.forcing_window(run)
t_on = a.attrs["wave_t_start"]
ALTS = {"O": [101, 150, 230], "H2": [101, 150, 460]}
rows = [("O", "num_dens"), ("H2", "num_dens"), ("O", "vel_rad"), ("H2", "vel_rad")]
fig, axes = plt.subplots(4, 3, figsize=(12, 9), sharex=True, sharey="row")
alt_cells = a.altitude.values / 1e3
for r, (sp, f) in enumerate(rows):
    absolute = f == "vel_rad"
    ser, fit = wave.perturbation_series(run, win, ALTS[sp], field=f, absolute=absolute, fit=True)
    for k in range(ser.sizes["radial"]):
        ax = axes[r, k]
        y, yf = ser.sel(species=sp).isel(radial=k), fit.sel(species=sp).isel(radial=k)
        ax.plot(y.time / 1e3, y, color=COL[sp], lw=0.6, alpha=0.45)
        ax.plot(yf.time / 1e3, yf, color=COL[sp], lw=2)
        ax.axvline(t_on / 1e3, color=REF, ls=":", lw=1.2); ax.axhline(0, color=REF, lw=0.8)
        z = float(ser.altitude_km[k]); i = int(np.argmin(abs(alt_cells - z)))
        N = float(a.wave_background_num_part.sel(species=sp).isel(radial=i))
        amp, se = (float(a[f"wave_{f}_amp"].sel(species=sp).isel(radial=i)), float(a[f"wave_{f}_amp_se"].sel(species=sp).isel(radial=i)))
        txt = (f"{sp} {z:.0f} km, N̄={N:.0f} (1/√N={1/np.sqrt(N):.2f})" if not absolute else f"{sp} {z:.0f} km, N̄={N:.0f}")
        txt += f"\namp {amp:.2f}±{se:.2f}" if not absolute else f"\namp {amp:.0f}±{se:.0f} m/s"
        if not np.isfinite(amp):
            txt += " (masked: N̄ < 10)"
        ax.text(0.02, 0.96, txt, transform=ax.transAxes, va="top", fontsize=8, color="#2b2b29")
    axes[r, 0].set_ylabel("δv_r [m/s]" if absolute else "δn/n̄")
axes[0, 0].set_ylim(-0.6, 0.6); axes[1, 0].set_ylim(-0.6, 0.6)
axes[2, 0].set_ylim(-250, 250); axes[3, 0].set_ylim(-250, 250)
for ax in axes[-1]:
    ax.set_xlim(8.5, run.t_end / 1e3); ax.set_xlabel("time [ks]")
fig.suptitle("OH2_A perturbations at selected altitudes: raw 4.8 s windows (light), harmonic + trend fit over 4 forcing periods (bold); dotted: forcing start",
             x=0.01, ha="left", fontsize=10)
fig.tight_layout(); fig.savefig(FIG / "fig4a_wave_series.png"); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(15, 5), sharey=True)
alt = a.altitude / 1e3
panels = [("num_dens", "|δn/n̄| at ω"), ("temp_trans_corr", "|δT/T̄| at ω"), ("vel_rad", "|δv_r| at ω [m/s]")]
for ax, (f, xl) in zip(axes, panels):
    for sp in ["O", "H2"]:
        amp, se = a[f"wave_{f}_amp"].sel(species=sp), a[f"wave_{f}_amp_se"].sel(species=sp)
        null = a[f"wave_{f}_null_amp"].sel(species=sp)
        ok = np.isfinite(null.values)
        ax.fill_betweenx(alt.values[ok], 0, null.values[ok], color=COL[sp], alpha=0.15, lw=0,
                         label=f"{sp} noise floor")
        ax.errorbar(amp, alt, xerr=se, color=COL[sp], marker="o", ms=4, lw=1.2, label=f"{sp} response")
    ax.set_xlabel(xl); ax.set_xlim(left=0)
for sp in ["O", "H2"]:
    for f, mk in (("num_dens", "o"), ("vel_rad", "s")):
        amp, se = a[f"wave_{f}_amp"].sel(species=sp), a[f"wave_{f}_amp_se"].sel(species=sp)
        sig = (amp / se > 2) & np.isfinite(a[f"wave_{f}_phase_rel_forcing"].sel(species=sp))
        ph = a[f"wave_{f}_phase_rel_forcing"].sel(species=sp).where(sig, drop=True)
        phe = a[f"wave_{f}_phase_se"].sel(species=sp).where(sig, drop=True)
        hollow = f == "vel_rad"
        axes[3].errorbar(np.unwrap(ph), alt.sel(radial=ph.radial), xerr=phe, color=COL[sp], marker=mk, ms=4.5, lw=1.2,
                         mfc="white" if hollow else COL[sp], ls="--" if hollow else "-",
                         label=f"{sp} {'δv_r' if hollow else 'δn'}")
    for ax in axes:
        ax.axhline(float(a.exobase_altitude.sel(species=sp)), color=COL[sp], lw=0.8, ls=":")
axes[3].set_xlabel("phase rel. to forcing [rad]")
axes[0].axvline(0.25, color=REF, ls="--", lw=1); axes[0].text(0.255, 105, "forcing 0.25", color=REF, fontsize=8)
axes[0].set_yscale("log"); axes[0].set_ylim(100, 5000); axes[0].set_ylabel("altitude [km]")
axes[0].legend(fontsize=7.5, loc="upper right"); axes[3].legend(fontsize=7.5, loc="lower left")
titles = ["density amplitude ±1σ", "temperature amplitude ±1σ", "vertical velocity amplitude ±1σ",
          "phase ±1σ where amp > 2σ (neg. = lags)"]
for ax, ti in zip(axes, titles):
    ax.set_title(ti, loc="left", fontsize=10)
fig.text(0.01, 0.005, "Shaded: noise floor (same fit over the 4 periods before forcing). Dotted: exobase (Kn = 1) per species.",
         fontsize=8, color=REF)
fig.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(FIG / "fig4b_wave_profiles.png"); plt.show()
display(pd.Series({k.replace("wave_", ""): v for k, v in a.attrs.items() if k.startswith("wave_")}))
print("inventory amplitude O, H2:", a.wave_inventory_amp.values, "+-", a.wave_inventory_amp_se.values)

/var/folders/n6/0qs5g9yx36v79k6jdbtt7b340000gq/T/ipykernel_48856/1464005553.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "fig4a_wave_series.png"); plt.show()


/var/folders/n6/0qs5g9yx36v79k6jdbtt7b340000gq/T/ipykernel_48856/1464005553.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(FIG / "fig4b_wave_profiles.png"); plt.show()


t_start             9849.395234
period               661.387927
omega                    0.0095
n_periods                   4.0
t_end              12494.946942
periods_in_run         4.173557
background_t0       6849.395234
background_t1       9849.395234
n_outputs                   548
min_part                   10.0
absolute_fields         vel_rad
null_t0             7203.843525
null_t1             9849.395234
null_n_periods                4
dtype: object

inventory amplitude O, H2: [0.10195705 0.03544223] +- [0.01122758 0.00485562]
